In [ ]:
%load_ext autoreload
%autoreload 2
from manager_tenure import features
from manager_tenure.config import RAW_DATA_DIR
from manager_tenure.dataset import find_latest_snapshot
snap = find_latest_snapshot(RAW_DATA_DIR)

import pandas as pd
import janitor

In [ ]:
snap = find_latest_snapshot(RAW_DATA_DIR)
spells_table = pd.read_html(snap, match = "Duration")[0]
spells_table = spells_table.drop(columns=["Nat.", "Years in League", "Ref."])
spells_table = spells_table.clean_names(strip_underscores=True)
spells_table


# Exploring the Premier League manager spells

This notebook explores the **cleaned** table that `make data` produces (`data/processed/spells.parquet`).
Each row is one managerial spell: one manager at one club, from `start_date` to `end_date`.

The sections are:
1. Load the table and check its shape
2. Quick looks at the data and its quirks
3. How long do spells last?
4. A closer look at **Southampton**
5. Checking the Premier League tenure column against the raw Wikipedia labels

Run the cells in order from the top. Each code cell has a short explanation above it.

## 1. Load the table
We read the parquet file the pipeline wrote, then check its size and column types.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from manager_tenure.config import SPELLS_FILE

spells = pd.read_parquet(SPELLS_FILE)
print(f"{len(spells)} spells, {spells['name'].nunique()} different managers, {spells['club'].nunique()} clubs")
spells.dtypes

## 2. Quick looks and data quirks
`head()` shows the first rows. `describe()` gives summary statistics for the numbers (like R's `summary()`).
Then we count the things the cleaning step flagged: caretakers, ongoing spells and missing nationalities.

In [ ]:
spells.head(3)

In [ ]:
spells[["tenure_days", "pl_tenure_days", "duration_days"]].describe().round(0)

In [ ]:
# Each flag is True/False, so the sum counts the Trues.
print("Caretaker spells:  ", spells["is_caretaker"].sum())
print("Ongoing spells:    ", spells["is_ongoing"].sum())
print("Missing nationality:", spells["nationality"].isna().sum())
print("Spells with zero Premier League days:", (spells["pl_tenure_days"] == 0).sum())

## 3. How long do spells last?
Tenure is measured two ways:
- `tenure_days`: total time in charge
- `pl_tenure_days`: days in the club's Premier League seasons

Both are in days. Dividing by 365.25 gives years. The histogram compares them.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = range(0, 9000, 365)  # one bin per year
ax.hist(spells["tenure_days"] / 365.25, bins=bins, alpha=0.6, label="Total tenure")
ax.hist(spells["pl_tenure_days"] / 365.25, bins=bins, alpha=0.6, label="Premier League tenure")
ax.set_xlabel("Years")
ax.set_ylabel("Number of spells")
ax.set_xlim(0, 25)
ax.legend()
ax.set_title("Most spells are short: a few long ones stand out")
plt.show()

In [ ]:
# The longest spells, total tenure.
spells.nlargest(10, "tenure_days")[["name", "club", "start_date", "end_date", "tenure_days"]]

## 4. A closer look at Southampton
We filter the table to every Southampton spell. Caretaker spells are included, so the
list shows every change in charge, however short.

In [ ]:
sot = spells[spells["club"] == "Southampton"].sort_values("start_date")
print(f"{len(sot)} spells under {sot['name'].nunique()} managers")
sot[["name", "start_date", "end_date", "is_caretaker", "tenure_days", "pl_tenure_days"]]

In [ ]:
# One row per manager: how many spells, and how long they were in charge in total.
# groupby() then agg() is the pandas version of group_by() %>% summarise() in dplyr.
by_manager = (
    sot.groupby("name")
    .agg(
        spells=("name", "size"),
        total_days=("tenure_days", "sum"),
        pl_days=("pl_tenure_days", "sum"),
        first_start=("start_date", "min"),
    )
    .sort_values("total_days", ascending=False)
)
by_manager["total_years"] = (by_manager["total_days"] / 365.25).round(1)
by_manager.head(10)

In [ ]:
# A timeline: each bar is one spell. Caretaker spells are hatched, so they stand out.
fig, ax = plt.subplots(figsize=(10, 9))
for i, row in enumerate(sot.itertuples()):
    ax.barh(
        y=i,
        width=(row.end_date - row.start_date).days,
        left=row.start_date,
        height=0.7,
        color="tab:grey" if row.is_caretaker else "tab:red",
        hatch="//" if row.is_caretaker else None,
        edgecolor="white",
    )
ax.set_yticks(range(len(sot)))
ax.set_yticklabels(sot["name"])
ax.invert_yaxis()  # oldest at the top
ax.set_title("Southampton managers, 1991 to 2025 (grey hatched = caretaker)")
ax.xaxis_date()
plt.tight_layout()
plt.show()

## 5. Can we trust the Premier League column?
The Premier League tenure comes from the "Years in League" column on Wikipedia. Those labels are
written by hand, so they don't always follow one rule. The cells below check Southampton.

- A **range** such as `2012–2013` works well. It is the start year of each season it covers.
- A **single year** such as `2004` is less clear: sometimes it means the season that ends in that year.
- A spell that **crosses two seasons** only gets the season(s) Wikipedia listed, so it can be undercounted.

In [ ]:
from manager_tenure.config import RAW_DATA_DIR
raw = pd.read_html(RAW_DATA_DIR / "wiki_premier_league_managers_2026-11-23.html", match="Duration")[0]
check = raw.loc[raw["Club"] == "Southampton", ["Name", "From", "Until", "Years in League"]]
check.merge(
    sot[["name", "pl_tenure_days"]].rename(columns={"name": "Name"}),
    on="Name",
    how="left",
)

### Questions to think about
1. Which Southampton spells look right to you, and which look wrong? Use the football history you know.
2. Ivan Jurić's spell is labelled `2024–2025`. Is that the same kind of label as `2004`?
3. Mark Hughes (2018) and Russell Martin (2024) cross season boundaries. What should a Premier League
   number say for them?
4. Would a separate table of "which clubs were in the Premier League in each season" fix these problems?
   Where would you find one, and what would it cost us?

These are the questions for open question 8 in `docs/docs/open-questions.md`.